# DCGAN for Anime Face Generation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/dcgan-anime-faces/blob/main/notebooks/dcgan_anime_faces.ipynb)

A clean PyTorch implementation of a Deep Convolutional GAN (DCGAN) that learns to generate 64×64 anime faces from the Kaggle *Anime Face Dataset*. The notebook runs end-to-end on Google Colab: **Runtime → Run all** (GPU recommended).

## 1. Setup
Imports and dependencies. `kagglehub` downloads the dataset directly, so no manual upload is needed.

In [ ]:
%pip install -q kagglehub

import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import kagglehub
import numpy as np
import torch
import torch.nn as nn
from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

## 2. Configuration
All hyperparameters live in one frozen dataclass, so experiments are changed in a single place and nothing is hard-coded below.

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 64
    channels: int = 3
    latent_dim: int = 100
    g_features: int = 64
    d_features: int = 64
    batch_size: int = 128
    epochs: int = 30
    lr: float = 2e-4
    real_label: float = 0.9
    betas: tuple = (0.5, 0.999)
    num_workers: int = 2
    n_samples: int = 64
    kaggle_dataset: str = "splcher/animefacedataset"
    output_dir: Path = Path("outputs")


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg.output_dir.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")

## 3. Dataset Download (Kaggle)
Downloads the [Anime Face Dataset](https://www.kaggle.com/datasets/splcher/animefacedataset) (63,565 images) and caches it locally.

In [ ]:
data_dir = Path(kagglehub.dataset_download(cfg.kaggle_dataset))
print(f"Dataset path: {data_dir}")

## 4. Reproducibility
Seeds Python, NumPy and PyTorch so runs are comparable. GPU kernels are not fully deterministic, so small differences between runs are still possible.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)

## 5. Data
Images are resized and center-cropped to 64×64, randomly flipped horizontally for augmentation, and normalized to [-1, 1] to match the generator's `Tanh` output.

In [ ]:
class ImageFolderDataset(Dataset):
    """Loads every image found under a root folder (searched recursively)."""

    EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def __init__(self, root: Path, transform: transforms.Compose):
        self.paths = sorted(p for p in root.rglob("*") if p.suffix.lower() in self.EXTENSIONS)
        if not self.paths:
            raise FileNotFoundError(f"No images found in {root}")
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> torch.Tensor:
        with Image.open(self.paths[index]) as img:
            return self.transform(img.convert("RGB"))


transform = transforms.Compose([
    transforms.Resize(cfg.image_size, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.CenterCrop(cfg.image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),  # scale to [-1, 1] to match Tanh output
])

dataset = ImageFolderDataset(data_dir, transform)
loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=device.type == "cuda",
    drop_last=True,
)
print(f"Images: {len(dataset):,} | Batches per epoch: {len(loader):,}")

real_batch = next(iter(loader))
grid = make_grid(real_batch[:cfg.n_samples], nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Real samples")
plt.axis("off")
plt.show()

## 6. Models
Standard DCGAN: a transposed-convolution generator and a strided-convolution discriminator with batch normalization, initialized from N(0, 0.02) as in Radford et al. (2015). The discriminator returns logits; the sigmoid is folded into `BCEWithLogitsLoss` for numerical stability.

In [ ]:
class Generator(nn.Module):
    """Maps a latent vector (B, latent_dim, 1, 1) to an image (B, C, 64, 64)."""

    def __init__(self, latent_dim: int, features: int, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, features * 8, 4, 1, 0, bias=False),    # 4x4
            nn.BatchNorm2d(features * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(features * 8, features * 4, 4, 2, 1, bias=False),  # 8x8
            nn.BatchNorm2d(features * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(features * 4, features * 2, 4, 2, 1, bias=False),  # 16x16
            nn.BatchNorm2d(features * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(features * 2, features, 4, 2, 1, bias=False),      # 32x32
            nn.BatchNorm2d(features),
            nn.ReLU(True),
            nn.ConvTranspose2d(features, channels, 4, 2, 1),                      # 64x64
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class Discriminator(nn.Module):
    """Maps an image (B, C, 64, 64) to a real/fake logit (B,)."""

    def __init__(self, features: int, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, features, 4, 2, 1),                               # 32x32
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),              # 16x16
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),           # 8x8
            nn.BatchNorm2d(features * 4),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(features * 4, features * 8, 4, 2, 1, bias=False),           # 4x4
            nn.BatchNorm2d(features * 8),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(features * 8, 1, 4, 1, 0),                                  # 1x1
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)


def init_weights(module: nn.Module) -> None:
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, 0.0, 0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, 1.0, 0.02)
        nn.init.zeros_(module.bias)


generator = Generator(cfg.latent_dim, cfg.g_features, cfg.channels).to(device)
discriminator = Discriminator(cfg.d_features, cfg.channels).to(device)
generator.apply(init_weights)
discriminator.apply(init_weights)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Generator params:     {n_params(generator):,}")
print(f"Discriminator params: {n_params(discriminator):,}")

## 7. Training
Alternating updates: the discriminator learns to separate real from generated images, then the generator is updated to fool it. One-sided label smoothing (real label 0.9) keeps the discriminator from becoming overconfident. A fixed noise batch is rendered after every epoch to track progress.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generator.parameters(), lr=cfg.lr, betas=cfg.betas)
opt_d = torch.optim.Adam(discriminator.parameters(), lr=cfg.lr, betas=cfg.betas)

fixed_noise = torch.randn(cfg.n_samples, cfg.latent_dim, 1, 1, device=device)


def train_epoch() -> tuple[float, float]:
    generator.train()
    discriminator.train()
    d_total, g_total = 0.0, 0.0

    for real in loader:
        real = real.to(device, non_blocking=True)
        batch = real.size(0)
        ones = torch.ones(batch, device=device)
        zeros = torch.zeros(batch, device=device)
        smooth_real = torch.full((batch,), cfg.real_label, device=device)

        # Discriminator step: maximize log D(x) + log(1 - D(G(z)))
        noise = torch.randn(batch, cfg.latent_dim, 1, 1, device=device)
        fake = generator(noise)
        loss_d = (
            criterion(discriminator(real), smooth_real)
            + criterion(discriminator(fake.detach()), zeros)
        ) / 2
        opt_d.zero_grad(set_to_none=True)
        loss_d.backward()
        opt_d.step()

        # Generator step: maximize log D(G(z))
        loss_g = criterion(discriminator(fake), ones)
        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        opt_g.step()

        d_total += loss_d.item()
        g_total += loss_g.item()

    return d_total / len(loader), g_total / len(loader)


@torch.no_grad()
def generate_samples() -> torch.Tensor:
    generator.eval()
    return generator(fixed_noise).cpu()


def render_frame(label: str) -> Image.Image:
    """Renders the fixed-noise samples as a labeled image grid."""
    grid = make_grid(generate_samples(), nrow=8, normalize=True, value_range=(-1, 1))
    frame = to_pil_image(grid)
    ImageDraw.Draw(frame).text((8, 8), label, fill="white", stroke_width=2, stroke_fill="black")
    return frame


history = {"d_loss": [], "g_loss": []}
frames = [render_frame("Epoch 0")]  # untrained generator = start of the GIF

for epoch in range(1, cfg.epochs + 1):
    d_loss, g_loss = train_epoch()
    history["d_loss"].append(d_loss)
    history["g_loss"].append(g_loss)

    frame = render_frame(f"Epoch {epoch}")
    frame.save(cfg.output_dir / f"samples_epoch_{epoch:03d}.png")
    frames.append(frame)
    print(f"Epoch {epoch:03d}/{cfg.epochs} | D loss: {d_loss:.4f} | G loss: {g_loss:.4f}")

## 8. Evaluation
GAN losses do not measure image quality, so progress is judged visually. This cell plots both losses and saves a grid of final samples.

In [ ]:
final = make_grid(generate_samples(), nrow=8, normalize=True, value_range=(-1, 1))
to_pil_image(final).save(cfg.output_dir / "final_samples.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["d_loss"], label="Discriminator")
axes[0].plot(history["g_loss"], label="Generator")
axes[0].set(title="Training losses", xlabel="Epoch", ylabel="Loss")
axes[0].legend()

axes[1].imshow(final.permute(1, 2, 0))
axes[1].set_title("Generated samples (final epoch)")
axes[1].axis("off")

fig.tight_layout()
fig.savefig(cfg.output_dir / "training_summary.png", dpi=150)
plt.show()

## 9. Latent Space Interpolation
Blends two noise vectors to check that the generator learned a smooth latent space instead of memorizing training images.

In [ ]:
@torch.no_grad()
def interpolate(z_start: torch.Tensor, z_end: torch.Tensor, steps: int = 10) -> torch.Tensor:
    generator.eval()
    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1, 1, 1)
    z = (1 - alphas) * z_start + alphas * z_end
    return generator(z).cpu()


z_a, z_b = torch.randn(2, 1, cfg.latent_dim, 1, 1, device=device)
strip = make_grid(interpolate(z_a, z_b), nrow=10, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(14, 2))
plt.imshow(strip.permute(1, 2, 0))
plt.title("Latent interpolation")
plt.axis("off")
plt.show()

## 10. Save Artifacts
Saves model weights and loss history for later reuse.

In [ ]:
torch.save(generator.state_dict(), cfg.output_dir / "generator.pt")
torch.save(discriminator.state_dict(), cfg.output_dir / "discriminator.pt")
np.savez(cfg.output_dir / "history.npz", **history)

print("Saved artifacts:")
for path in sorted(cfg.output_dir.glob("*.pt")) + sorted(cfg.output_dir.glob("*.npz")):
    print(f"  {path}")

## 11. Generation Progress (GIF)
Combines the per-epoch sample grids into an animation, from the untrained generator to the final epoch.

In [ ]:
FRAME_MS = 150
FINAL_HOLD_MS = 2500

gif_path = cfg.output_dir / "generation_progress.gif"
frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=[FRAME_MS] * (len(frames) - 1) + [FINAL_HOLD_MS],
    loop=0,
)
print(f"Saved: {gif_path} ({gif_path.stat().st_size / 1e6:.1f} MB, {len(frames)} frames)")
display(IPyImage(filename=str(gif_path)))

## 12. Export for GitHub
Downloads the GIF shown in the README. Save it as `assets/generation_progress.gif` in the repository.

In [ ]:
from google.colab import files

files.download(str(cfg.output_dir / "generation_progress.gif"))